# 03 - Downstream evaluation: AlphaEarth vs. BEACON

Compare frozen **AlphaEarth** embeddings (64 dimensions) and **BEACON** embeddings (128 dimensions) on nine Houston downstream tasks.

| Task | Spatial support | Label source | Metric |
|---|---|---|---|
| Median household income | Census block group | ACS | R² |
| Obesity and poor mental health | Census tract | CDC PLACES | R² |
| Nighttime lights | VIIRS pixel | VNL 2024 | R² |
| Daytime and nighttime land-surface temperature | MODIS pixel | 2024 annual mean LST in °C | R² |
| PM2.5 | SatPM pixel | V6GL03 CNN PM2.5, 2024 | R² |
| Land cover | 300,000 fixed 10 m samples | NLCD 2024 | Macro-F1 |
| Functional land use | Parcel | City of Houston land-use map | Macro-F1 |

**Evaluation protocol:** compare both representations on identical samples and identical 70/10/20 train-validation-test splits. Stratify classification tasks. Fit feature and target transformations on training data only. Evaluate linear and one-hidden-layer MLP probes with Adam, validation-based early stopping, and five random seeds.

**Workflow:** aggregate both embeddings to each task's spatial support, train frozen probes, and summarize paired results.


## 0. Configuration and shared utilities


In [ ]:
from pathlib import Path
import os
from contextlib import ExitStack
import copy
import gc
import random

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import requests
from rasterio.crs import CRS
from rasterio.enums import Resampling
from rasterio.features import geometry_mask, rasterize
from rasterio.transform import rowcol, xy
from rasterio.vrt import WarpedVRT
from rasterio.windows import Window, bounds as window_bounds
from pyproj import Transformer
from shapely.geometry import box, mapping
from tqdm.auto import tqdm

PROJECT_DIR = Path(os.environ.get("BEACON_PROJECT_DIR", Path.cwd())).expanduser().resolve()
DOWNSTREAM_DIR = PROJECT_DIR / "Downstream"
FEATURE_DIR = DOWNSTREAM_DIR / "features"       # 64- and 128-dimensional feature tables for each task
RESULT_DIR = DOWNSTREAM_DIR / "results"

# ---- Embedding rasters on matched grids ----
AE64_DIR = PROJECT_DIR / "Houston_AE_2024"
AE128_DIR = PROJECT_DIR / "contrastive_learning" / "houston_beacon_embeddings_128"

# ---- Boundaries and labels ----
CBG_GPKG = PROJECT_DIR / "houston_msa" / "houston_msa_2024_cbg.gpkg"
TRACT_GPKG = PROJECT_DIR / "houston_msa" / "houston_msa_2024_tract.gpkg"
CBG_LABELS = DOWNSTREAM_DIR / "houston_downstream_CBG_level_merged.csv"      # Includes geoid and median_household_income
TRACT_LABELS = DOWNSTREAM_DIR / "houston_downstream_Tract_level_merged.csv"  # Includes geoid, pct_obesity, and pct_mental_health

NTL_PATH = PROJECT_DIR / "VNL_npp_2024" / "VNL_npp_2024_global_vcmslcfg_v2_c202502261200.average_masked.dat.tif"
LST_DAY_PATH = PROJECT_DIR / "LST_2024" / "Houston_2024_Annual_Mean_LST_Day_MODIS.tif"
LST_NIGHT_PATH = PROJECT_DIR / "LST_2024" / "Houston_2024_Annual_Mean_LST_Night_Aqua.tif"
PM25_NC_PATH = PROJECT_DIR / "PM25_2024" / "V6GL03.CNNPM25.NA.202401-202412.nc"
NLCD_PATH = PROJECT_DIR / "Annual_NLCD_LndCov_2024" / "Annual_NLCD_LndCov_2024_CU_C1V1.tif"
LANDUSE_DIR = PROJECT_DIR / "houston_land_use"
LANDUSE_URL = "https://mycity2.houstontx.gov/pubgis02/rest/services/HoustonMap/Landuse/MapServer/0/query"

REPRESENTATIONS = {"AlphaEarth": 64, "BEACON": 128}

for d in [FEATURE_DIR, RESULT_DIR, LANDUSE_DIR]:
    d.mkdir(parents=True, exist_ok=True)


# ============================================================
# Shared utilities
# ============================================================

def feature_path(task, dim):
    return FEATURE_DIR / f"{task}_ae{dim}.csv"


def grid_signature(path):
    with rasterio.open(path) as src:
        return (src.crs.to_string() if src.crs else None, src.width, src.height,
                tuple(round(v, 12) for v in tuple(src.transform)[:6]))


def paired_tiles():
    """Pair 64- and 128-band tiles by their grid signatures."""
    maps = [{grid_signature(p): p for p in sorted(folder.rglob("*.tif"))} for folder in (AE64_DIR, AE128_DIR)]
    if set(maps[0]) != set(maps[1]):
        raise ValueError("The 64- and 128-band raster grids do not match")
    return [(maps[0][k], maps[1][k]) for k in sorted(maps[0])]


def valid_mask(data, nodata=None):
    ok = np.isfinite(data).all(axis=0)
    if nodata is not None:
        ok &= ~np.isclose(data, nodata).any(axis=0)
    return ok


_tracts = gpd.read_file(TRACT_GPKG)


def houston_geometry(crs):
    """Return the Houston MSA boundary as the union of all census tracts."""
    g = _tracts.to_crs(crs).geometry
    return g.union_all() if hasattr(g, "union_all") else g.unary_union


def group_sum(inverse, n_groups, values):
    """Sum values with shape [N, D] by group to produce [n_groups, D]."""
    return np.stack([np.bincount(inverse, weights=values[:, b], minlength=n_groups)
                     for b in range(values.shape[1])], axis=1)


def save_features(task, meta, means):
    """Save one feature table per representation with metadata and ae_00 through ae_{D-1}."""
    for dim, X in means.items():
        cols = [f"ae_{i:02d}" for i in range(X.shape[1])]
        out = pd.concat([meta.reset_index(drop=True), pd.DataFrame(X.astype(np.float32), columns=cols)], axis=1)
        out.to_csv(feature_path(task, dim), index=False)
        print(f"saved {feature_path(task, dim).name}: {out.shape}")

## 1. Aggregate embeddings to downstream spatial supports

Read AlphaEarth and BEACON from matched grids and use only pixels that are valid in both representations. This guarantees that paired features for each sample use the same pixel support.


### 1.1 Polygon supports: census block groups, census tracts, and land-use parcels

Average all 10 m embedding pixels whose centers fall inside each polygon.


In [ ]:
def pool_polygons(polygons, stripe_rows=256):
    """Pool matched raster pixels inside each polygon and return counts and mean embeddings."""
    pairs = paired_tiles()
    with rasterio.open(pairs[0][0]) as s64, rasterio.open(pairs[0][1]) as s128:
        crs, b64, b128 = s64.crs, s64.count, s128.count

    geoms = polygons.to_crs(crs).geometry.reset_index(drop=True)
    sindex = geoms.sindex
    n = len(geoms)
    counts = np.zeros(n, dtype=np.int64)
    sums = {64: np.zeros((n, b64)), 128: np.zeros((n, b128))}

    for p64, p128 in tqdm(pairs, desc="tiles"):
        with rasterio.open(p64) as s64, rasterio.open(p128) as s128:
            if len(sindex.query(box(*s64.bounds), predicate="intersects")) == 0:
                continue
            for row_off in range(0, s64.height, stripe_rows):
                window = Window(0, row_off, s64.width, min(stripe_rows, s64.height - row_off))
                idx = sindex.query(box(*window_bounds(window, s64.transform)), predicate="intersects")
                if len(idx) == 0:
                    continue
                labels = rasterize(
                    ((geoms.iloc[i], k + 1) for k, i in enumerate(idx)),
                    out_shape=(window.height, window.width),
                    transform=s64.window_transform(window), fill=0, dtype="int32",
                )
                if labels.max() == 0:
                    continue
                d64 = s64.read(window=window, out_dtype="float32")
                d128 = s128.read(window=window, out_dtype="float32")
                ok = (labels > 0) & valid_mask(d64, s64.nodata) & valid_mask(d128, s128.nodata)
                if not ok.any():
                    continue
                local = labels[ok] - 1
                counts[idx] += np.bincount(local, minlength=len(idx))
                sums[64][idx] += group_sum(local, len(idx), d64[:, ok].T)
                sums[128][idx] += group_sum(local, len(idx), d128[:, ok].T)

    with np.errstate(invalid="ignore", divide="ignore"):
        means = {dim: s / counts[:, None] for dim, s in sums.items()}
    return counts, means

In [ ]:
# Census block groups and tracts: mean embeddings and labels
for unit, gpkg, labels_csv in [("cbg", CBG_GPKG, CBG_LABELS), ("tract", TRACT_GPKG, TRACT_LABELS)]:
    units = gpd.read_file(gpkg)[["GEOID", "geometry"]]
    units["GEOID"] = units["GEOID"].astype(str)
    counts, means = pool_polygons(units)

    labels = pd.read_csv(labels_csv, dtype={"geoid": str}, low_memory=False).rename(columns={"geoid": "GEOID"})
    meta = units[["GEOID"]].assign(n_pixels=counts).merge(labels, on="GEOID", how="left", validate="one_to_one")

    keep = counts > 0
    save_features(unit, meta[keep], {dim: X[keep] for dim, X in means.items()})

In [ ]:
# Functional land use: download Houston parcels, sample up to 4,000 per class, and pool embeddings
LANDUSE_PER_CLASS = 4000
landuse_raw = LANDUSE_DIR / "houston_landuse.parquet"

if not landuse_raw.exists():
    features, offset = [], 0
    while True:
        page = requests.get(LANDUSE_URL, params={
            "where": "1=1", "outFields": "*", "returnGeometry": "true", "f": "geojson",
            "resultOffset": offset, "resultRecordCount": 2000,
        }, timeout=120).json().get("features", [])
        if not page:
            break
        features.extend(page)
        offset += len(page)
    gpd.GeoDataFrame.from_features(features, crs="EPSG:4326").to_parquet(landuse_raw)

parcels = gpd.read_parquet(landuse_raw)
if parcels.crs is None:
    parcels = parcels.set_crs("EPSG:4326")
parcels = parcels[parcels["GROUP_DSCR"].notna() & parcels.geometry.notna() & ~parcels.geometry.is_empty]
parcels = pd.concat(
    [g.sample(n=min(len(g), LANDUSE_PER_CLASS), random_state=42) for _, g in parcels.groupby("GROUP_DSCR")],
    ignore_index=True,
)
parcels["sample_id"] = np.arange(len(parcels))
rep = parcels.geometry.representative_point()
parcels["lon"], parcels["lat"] = rep.x, rep.y
print(parcels["GROUP_DSCR"].value_counts())

counts, means = pool_polygons(parcels)
meta = parcels[["sample_id", "OBJECTID", "GROUP_DSCR", "LANDUSE_DSCR", "LANDUSE_CD", "lon", "lat"]].assign(n_pixels=counts)
keep = counts > 0
save_features("landuse", meta[keep], {dim: X[keep] for dim, X in means.items()})

### 1.2 Raster targets: nighttime lights, land-surface temperature, and PM2.5

Map every valid 10 m embedding pixel in the Houston MSA to its target raster cell, then average embeddings within each target cell. Retain cells with centers inside the Houston MSA and valid target values. All raster targets are accumulated in one pass.

| Target | Valid range |
|---|---:|
| Nighttime lights | >= 0 |
| LST in °C | -30 to 80 |
| PM2.5 in µg/m³ | 0 to 100 |

When the PM2.5 NetCDF omits a coordinate reference system, the code assumes EPSG:4326.


In [ ]:
def netcdf_variable(nc_path, keyword="PM25"):
    """Find the primary PM2.5 variable in a NetCDF file while excluding uncertainty variables."""
    with rasterio.open(nc_path) as src:
        subdatasets = src.subdatasets
    if not subdatasets:
        return str(nc_path)
    for sd in subdatasets:
        u = sd.upper()
        if keyword in u.replace("PM2_5", "PM25").replace("PM2.5", "PM25") and not any(
            w in u for w in ("SIGMA", "UNCERTAINT", "QFLAG")
        ):
            return sd
    raise ValueError(f"No {keyword} variable found in {nc_path}")


RASTER_TARGETS = {
    "ntl":       {"path": NTL_PATH,       "valid": (0, np.inf)},
    "lst_day":   {"path": LST_DAY_PATH,   "valid": (-30, 80)},
    "lst_night": {"path": LST_NIGHT_PATH, "valid": (-30, 80)},
    "pm25":      {"path": netcdf_variable(PM25_NC_PATH), "valid": (0, 100), "default_crs": "EPSG:4326"},
}


class PixelAccumulator:
    """Accumulate pixel counts and embedding sums by target raster cell."""

    def __init__(self):
        self.store = {}

    def add(self, keys, v64, v128):
        uniq, inv = np.unique(keys, return_inverse=True)
        c = np.bincount(inv, minlength=len(uniq))
        s64, s128 = group_sum(inv, len(uniq), v64), group_sum(inv, len(uniq), v128)
        for i, k in enumerate(uniq):
            if k in self.store:
                entry = self.store[k]
                entry[0] += c[i]
                entry[1] += s64[i]
                entry[2] += s128[i]
            else:
                self.store[k] = [int(c[i]), s64[i].copy(), s128[i].copy()]


def sample_target(src, xs, ys):
    """Sample target-cell centers, average multiple bands, and treat nodata as missing."""
    vals = np.asarray(list(src.sample(zip(xs, ys))), dtype=np.float32)
    if src.nodata is not None:
        vals[np.isclose(vals, src.nodata)] = np.nan
    with np.errstate(all="ignore"):
        return np.nanmean(vals, axis=1)


def pool_to_raster_targets(targets, stripe_rows=32):
    pairs = paired_tiles()
    with rasterio.open(pairs[0][0]) as s:
        ae_crs = s.crs
    houston_ae = houston_geometry(ae_crs)
    mask_mismatch = 0

    with ExitStack() as stack:
        states = {}
        for name, cfg in targets.items():
            src = stack.enter_context(rasterio.open(cfg["path"]))
            crs = src.crs or CRS.from_user_input(cfg["default_crs"])
            states[name] = {"src": src, "crs": crs, "acc": PixelAccumulator(),
                            "to_target": Transformer.from_crs(ae_crs, crs, always_xy=True)}

        # Accumulate embedding pixels into target raster cells
        for p64, p128 in tqdm(pairs, desc="tiles"):
            with rasterio.open(p64) as s64, rasterio.open(p128) as s128:
                if not box(*s64.bounds).intersects(houston_ae):
                    continue
                for row_off in range(0, s64.height, stripe_rows):
                    window = Window(0, row_off, s64.width, min(stripe_rows, s64.height - row_off))
                    if not box(*window_bounds(window, s64.transform)).intersects(houston_ae):
                        continue
                    inside = geometry_mask([mapping(houston_ae)], out_shape=(window.height, window.width),
                                           transform=s64.window_transform(window), invert=True)
                    if not inside.any():
                        continue
                    d64 = s64.read(window=window, out_dtype="float32")
                    d128 = s128.read(window=window, out_dtype="float32")
                    ok64, ok128 = valid_mask(d64, s64.nodata), valid_mask(d128, s128.nodata)
                    mask_mismatch += int((inside & (ok64 ^ ok128)).sum())
                    ok = inside & ok64 & ok128
                    if not ok.any():
                        continue
                    r, c = np.nonzero(ok)
                    xs, ys = xy(s64.transform, r + row_off, c, offset="center")
                    v64, v128 = d64[:, r, c].T, d128[:, r, c].T
                    for st in states.values():
                        tx, ty = st["to_target"].transform(np.asarray(xs), np.asarray(ys))
                        tr, tc = rowcol(st["src"].transform, tx, ty)
                        tr, tc = np.asarray(tr, dtype=np.int64), np.asarray(tc, dtype=np.int64)
                        inb = (tr >= 0) & (tr < st["src"].height) & (tc >= 0) & (tc < st["src"].width)
                        if inb.any():
                            st["acc"].add(tr[inb] * st["src"].width + tc[inb], v64[inb], v128[inb])

        if mask_mismatch:
            raise ValueError(f"Valid-pixel masks differ between the 64- and 128-band rasters: {mask_mismatch:,} pixels")

        # Filter valid target cells and save feature tables
        for name, st in states.items():
            src, store = st["src"], st["acc"].store
            keys = np.array(sorted(store), dtype=np.int64)
            rows, cols = keys // src.width, keys % src.width
            cx, cy = (np.asarray(v) for v in xy(src.transform, rows, cols, offset="center"))

            centers = gpd.GeoSeries(gpd.points_from_xy(cx, cy), crs=st["crs"])
            houston_t = houston_geometry(st["crs"])
            keep = np.array(centers.within(houston_t) | centers.touches(houston_t))

            values = np.full(len(keys), np.nan, dtype=np.float32)
            values[keep] = sample_target(src, cx[keep], cy[keep])
            lo, hi = targets[name]["valid"]
            keep &= np.isfinite(values) & (values >= lo) & (values <= hi)

            lon, lat = Transformer.from_crs(st["crs"], "EPSG:4326", always_xy=True).transform(cx[keep], cy[keep])
            meta = pd.DataFrame({
                f"{name}_row": rows[keep], f"{name}_col": cols[keep],
                "lon": lon, "lat": lat,
                f"{name}_value": values[keep],
                "n_pixels": [store[k][0] for k in keys[keep]],
            })
            means = {
                64: np.vstack([store[k][1] / store[k][0] for k in keys[keep]]),
                128: np.vstack([store[k][2] / store[k][0] for k in keys[keep]]),
            }
            save_features(name, meta, means)


pool_to_raster_targets(RASTER_TARGETS)
gc.collect()

### 1.3 Land cover: fixed NLCD samples

Reproject NLCD to each AlphaEarth tile with nearest-neighbor resampling. Assign random priorities to valid pixels within the Houston MSA and retain the 300,000 smallest priorities. Extract AlphaEarth and BEACON embeddings at exactly the same sample locations.


In [ ]:
LULC_N_SAMPLES = 300_000
LULC_SEED = 42


def build_lulc_samples(n_samples=LULC_N_SAMPLES, seed=LULC_SEED, stripe_rows=256):
    rng = np.random.default_rng(seed)
    tiles = sorted(AE64_DIR.rglob("*.tif"))
    with rasterio.open(tiles[0]) as s:
        houston = houston_geometry(s.crs)

    kept = None   # Retain only the n_samples candidates with the smallest priorities
    with rasterio.open(NLCD_PATH) as nlcd:
        for path in tqdm(tiles, desc="tiles"):
            with rasterio.open(path) as ae:
                if not box(*ae.bounds).intersects(houston):
                    continue
                with WarpedVRT(nlcd, crs=ae.crs, transform=ae.transform, width=ae.width,
                               height=ae.height, resampling=Resampling.nearest) as vrt:
                    for row_off in range(0, ae.height, stripe_rows):
                        window = Window(0, row_off, ae.width, min(stripe_rows, ae.height - row_off))
                        if not box(*window_bounds(window, ae.transform)).intersects(houston):
                            continue
                        band1 = ae.read(1, window=window, out_dtype="float32")
                        lulc = vrt.read(1, window=window)
                        ok = geometry_mask([mapping(houston)], out_shape=band1.shape,
                                           transform=ae.window_transform(window), invert=True)
                        ok &= np.isfinite(band1) & (lulc > 0)
                        if ae.nodata is not None:
                            ok &= ~np.isclose(band1, ae.nodata)
                        if nlcd.nodata is not None:
                            ok &= lulc != nlcd.nodata
                        r, c = np.nonzero(ok)
                        priority = rng.random(len(r))
                        if kept is not None and len(kept) >= n_samples:   # At capacity; accept only candidates with smaller priorities
                            sel = priority < kept["priority"].max()
                            r, c, priority = r[sel], c[sel], priority[sel]
                        if len(r) == 0:
                            continue
                        cand = pd.DataFrame({"priority": priority, "ref_tile": path.name,
                                             "ref_row": r + row_off, "ref_col": c,
                                             "lulc_class": lulc[r, c].astype(np.int16)})
                        kept = cand if kept is None else pd.concat([kept, cand], ignore_index=True)
                        if len(kept) > n_samples:
                            kept = kept.nsmallest(n_samples, "priority")

    samples = kept.sort_values(["ref_tile", "ref_row", "ref_col"]).reset_index(drop=True)
    samples.insert(0, "sample_id", np.arange(len(samples)))
    return samples.drop(columns="priority")


def extract_at_samples(samples):
    """Read both representations at fixed sample locations and convert coordinates to longitude and latitude."""
    pair_by_name = {p64.name: (p64, p128) for p64, p128 in paired_tiles()}
    out = {64: [], 128: []}
    meta = []
    for tile, g in tqdm(samples.groupby("ref_tile", sort=False), desc="tiles"):
        p64, p128 = pair_by_name[tile]
        with rasterio.open(p64) as s64, rasterio.open(p128) as s128:
            xs, ys = xy(s64.transform, g["ref_row"].to_numpy(), g["ref_col"].to_numpy(), offset="center")
            xs, ys = np.asarray(xs), np.asarray(ys)
            lon, lat = Transformer.from_crs(s64.crs, "EPSG:4326", always_xy=True).transform(xs, ys)
            for dim, src in [(64, s64), (128, s128)]:
                v = np.asarray(list(src.sample(zip(xs, ys))), dtype=np.float32)
                if not valid_mask(v.T, src.nodata).all():
                    raise ValueError(f"{tile}: Invalid embedding at a fixed sample location for the {dim}-dimensional representation")
                out[dim].append(v)
        meta.append(g.assign(x=xs, y=ys, lon=lon, lat=lat))
    return pd.concat(meta, ignore_index=True), {dim: np.vstack(v) for dim, v in out.items()}


lulc_samples = build_lulc_samples()
print(f"LULC samples: {len(lulc_samples):,}")
print(lulc_samples["lulc_class"].value_counts().sort_index())
meta, means = extract_at_samples(lulc_samples)
save_features("lulc", meta, means)

## 2. Frozen-probe evaluation

For each task, retain samples with valid targets and both representations. Use identical splits for AlphaEarth and BEACON under the same random seed.

- Regression: standardize the target with training-set statistics, optimize MSE, stop on validation loss, and report R², RMSE, and MAE in the original target units.
- Classification: use clipped inverse-frequency class weights and stop on validation Macro-F1, with validation loss as the tie breaker.


In [ ]:
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (r2_score, mean_squared_error, mean_absolute_error,
                             accuracy_score, balanced_accuracy_score, f1_score)

SEEDS = [0, 1, 2, 3, 4]
PROBES = ["Linear", "MLP"]
EPOCHS = 500
PATIENCE = 10
LR = 1e-3
WEIGHT_DECAY = 1e-4
MLP_HIDDEN, MLP_DROPOUT = 128, 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH_SIZE = 1024 if DEVICE == "cuda" else 512

# Task definition: feature table, sample key, target column, and task type
TASKS = {
    "Income":        ("cbg",       ["GEOID"],                     "median_household_income", "regression"),
    "Obesity":       ("tract",     ["GEOID"],                     "pct_obesity",             "regression"),
    "Mental health": ("tract",     ["GEOID"],                     "pct_mental_health",       "regression"),
    "NTL":           ("ntl",       ["ntl_row", "ntl_col"],        "ntl_value",               "regression"),
    "LST day":       ("lst_day",   ["lst_day_row", "lst_day_col"],     "lst_day_value",      "regression"),
    "LST night":     ("lst_night", ["lst_night_row", "lst_night_col"], "lst_night_value",    "regression"),
    "PM2.5":         ("pm25",      ["pm25_row", "pm25_col"],      "pm25_value",              "regression"),
    "Land cover":    ("lulc",      ["sample_id"],                 "lulc_class",              "classification"),
    "Land use":      ("landuse",   ["sample_id"],                 "GROUP_DSCR",              "classification"),
}


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def load_task(table, keys, target, kind):
    """Load both representations, align common samples, and return feature matrices and targets."""
    frames = {}
    for rep, dim in REPRESENTATIONS.items():
        df = pd.read_csv(feature_path(table, dim), dtype={"GEOID": str}, low_memory=False)
        if kind == "regression":
            df = df[pd.to_numeric(df[target], errors="coerce").notna()]
        else:
            df = df[df[target].notna()]
        frames[rep] = df.set_index(keys).sort_index()

    common = frames["AlphaEarth"].index.intersection(frames["BEACON"].index)
    frames = {rep: df.loc[common] for rep, df in frames.items()}

    y_a = frames["AlphaEarth"][target].astype(str)
    y_b = frames["BEACON"][target].astype(str)
    if not y_a.equals(y_b):
        raise ValueError(f"{table}: Target labels differ between representations")

    X = {rep: frames[rep][[f"ae_{i:02d}" for i in range(dim)]].to_numpy(np.float32)
         for rep, dim in REPRESENTATIONS.items()}
    y = frames["AlphaEarth"][target]
    y = y.astype(float).to_numpy() if kind == "regression" else LabelEncoder().fit_transform(y.astype(str))
    return X, y


def make_split(y, seed, stratify):
    idx = np.arange(len(y))
    tr, tmp = train_test_split(idx, test_size=0.30, random_state=seed, stratify=y if stratify else None)
    va, te = train_test_split(tmp, test_size=2 / 3, random_state=seed, stratify=y[tmp] if stratify else None)
    return np.sort(tr), np.sort(va), np.sort(te)


def scale_features(X, tr, va, te):
    """Impute non-finite values with training medians and standardize with training statistics."""
    X = np.where(np.isfinite(X), X, np.nan)
    med = np.nan_to_num(np.nanmedian(X[tr], axis=0))
    X = np.where(np.isnan(X), med, X)
    sc = StandardScaler().fit(X[tr])
    return [sc.transform(X[i]).astype(np.float32) for i in (tr, va, te)]


def make_probe(name, d_in, d_out):
    if name == "Linear":
        return nn.Linear(d_in, d_out)
    return nn.Sequential(nn.Linear(d_in, MLP_HIDDEN), nn.ReLU(), nn.Dropout(MLP_DROPOUT), nn.Linear(MLP_HIDDEN, d_out))


def train_probe(probe, X_tr, y_tr, X_va, y_va, X_te, kind, n_out, seed, class_weight=None):
    set_seed(seed)
    model = make_probe(probe, X_tr.shape[1], n_out).to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    y_dtype = torch.float32 if kind == "regression" else torch.long
    if kind == "regression":
        loss_fn = nn.MSELoss()
    else:
        w = None if class_weight is None else torch.tensor(class_weight, device=DEVICE)
        loss_fn = nn.CrossEntropyLoss(weight=w)

    g = torch.Generator().manual_seed(seed)
    loader = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(torch.tensor(X_tr), torch.tensor(y_tr, dtype=y_dtype)),
        batch_size=BATCH_SIZE, shuffle=True, generator=g,
    )
    X_va_t = torch.tensor(X_va, device=DEVICE)
    y_va_t = torch.tensor(y_va, dtype=y_dtype, device=DEVICE)

    best = (-np.inf, np.inf)   # (score, validation loss); regression uses validation loss only
    best_state, best_epoch, wait = None, 0, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb in loader:
            opt.zero_grad()
            loss_fn(model(xb.to(DEVICE)), yb.to(DEVICE)).backward()
            opt.step()

        model.eval()
        with torch.no_grad():
            out = model(X_va_t)
            val_loss = loss_fn(out, y_va_t).item()
        score = 0.0 if kind == "regression" else f1_score(y_va, out.argmax(1).cpu().numpy(), average="macro", zero_division=0)

        if score > best[0] + 1e-8 or (np.isclose(score, best[0]) and val_loss < best[1]):
            best, best_state, best_epoch, wait = (score, val_loss), copy.deepcopy(model.state_dict()), epoch, 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(torch.tensor(X_te, device=DEVICE)).cpu().numpy()
    return pred, best_epoch


def evaluate_task(task):
    table, keys, target, kind = TASKS[task]
    X_all, y = load_task(table, keys, target, kind)
    n_classes = int(y.max()) + 1 if kind == "classification" else 1
    print(f"\n{task}: n = {len(y):,}" + (f", classes = {n_classes}" if n_classes > 1 else ""))

    records = []
    for seed in SEEDS:
        tr, va, te = make_split(y, seed, stratify=(kind == "classification"))
        if kind == "regression":
            y_sc = StandardScaler().fit(y[tr, None])
            y_tr, y_va = (y_sc.transform(y[i, None]).astype(np.float32) for i in (tr, va))
            class_weight = None
        else:
            y_tr, y_va = y[tr], y[va]
            counts = np.bincount(y_tr, minlength=n_classes).astype(float)
            class_weight = np.clip(len(y_tr) / (n_classes * counts), 0.2, 10.0).astype(np.float32)

        for rep, X in X_all.items():
            X_tr, X_va, X_te = scale_features(X, tr, va, te)
            for probe in PROBES:
                pred, epoch = train_probe(probe, X_tr, y_tr, X_va, y_va, X_te, kind, n_classes, seed, class_weight)
                rec = {"task": task, "representation": rep, "probe": probe, "seed": seed,
                       "n_samples": len(y), "best_epoch": epoch}
                if kind == "regression":
                    p = y_sc.inverse_transform(pred).ravel()
                    rec.update(r2=r2_score(y[te], p), rmse=np.sqrt(mean_squared_error(y[te], p)),
                               mae=mean_absolute_error(y[te], p))
                else:
                    p = pred.argmax(1)
                    rec.update(accuracy=accuracy_score(y[te], p),
                               balanced_accuracy=balanced_accuracy_score(y[te], p),
                               macro_f1=f1_score(y[te], p, average="macro", zero_division=0),
                               weighted_f1=f1_score(y[te], p, average="weighted", zero_division=0))
                records.append(rec)
                main = rec.get("r2", rec.get("macro_f1"))
                print(f"  seed {seed} | {rep:<7} | {probe:<6} | {'R2' if kind == 'regression' else 'F1'} = {main:.4f}")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    return records

In [ ]:
all_records = []
for task in TASKS:
    all_records += evaluate_task(task)

detail = pd.DataFrame(all_records)
detail["metric"] = np.where(detail["r2"].notna(), "R2", "Macro-F1")
detail["score"] = detail["r2"].fillna(detail["macro_f1"])
detail.to_csv(RESULT_DIR / "probe_results_detail.csv", index=False)

## 3. Result summary

Use R² as the primary regression metric and Macro-F1 as the primary classification metric. Compute `delta_score = BEACON - AlphaEarth` and apply paired t-tests across seeds because each representation uses the same split for a given seed.


In [ ]:
from scipy.stats import ttest_rel
import matplotlib.pyplot as plt

summary = (
    detail.groupby(["task", "metric", "probe", "representation"], sort=False)["score"]
    .agg(["mean", "std"]).reset_index()
)
table = summary.pivot_table(index=["task", "metric", "probe"], columns="representation",
                            values=["mean", "std"], sort=False)

rows = []
for (task, probe), g in detail.groupby(["task", "probe"], sort=False):
    a = g[g["representation"] == "AlphaEarth"].sort_values("seed")["score"].to_numpy()
    b = g[g["representation"] == "BEACON"].sort_values("seed")["score"].to_numpy()
    rows.append({"task": task, "metric": g["metric"].iloc[0], "probe": probe,
                 "AlphaEarth": f"{a.mean():.4f} ± {a.std(ddof=1):.4f}",
                 "BEACON": f"{b.mean():.4f} ± {b.std(ddof=1):.4f}",
                 "delta": b.mean() - a.mean(),
                 "relative_%": 100 * (b.mean() - a.mean()) / abs(a.mean()),
                 "p_value": ttest_rel(b, a).pvalue})
comparison = pd.DataFrame(rows)
comparison.to_csv(RESULT_DIR / "probe_results_beacon_vs_ae_only.csv", index=False)
comparison.round(4)

In [ ]:
fig, axes = plt.subplots(1, len(PROBES), figsize=(6 * len(PROBES), 4.5), sharey=True)
for ax, probe in zip(np.atleast_1d(axes), PROBES):
    c = comparison[comparison["probe"] == probe]
    colors = ["#1b9e77" if d > 0 else "#d95f02" for d in c["delta"]]
    ax.barh(c["task"], c["relative_%"], color=colors)
    ax.axvline(0, color="black", lw=0.8)
    for y, (v, p) in enumerate(zip(c["relative_%"], c["p_value"])):
        stars = "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 0.05 else ""
        ax.text(v, y, f" {stars}", va="center", ha="left" if v >= 0 else "right")
    ax.set(title=f"{probe} probe", xlabel="BEACON vs AlphaEarth (relative change, %)")
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(RESULT_DIR / "beacon_vs_ae_only.png", dpi=200, bbox_inches="tight")
plt.show()